In [0]:
# ============================================================
# NOTEBOOK: nb_01_CustomerMaster
# PURPOSE:  Replaces SQL Procedure 1 (usp_LoadCustomerMaster)
#           Reads 7 source systems, deduplicates, cleanses,
#           and MERGEs into warehouse.customer_master.
# LANGUAGE: PySpark (Python)
# CATALOG:  retailbank_dev
# ============================================================

# --------------------------------------------------------
# IMPORTS
# uuid      = creates unique GUIDs (like NEWID() in SQL)
# datetime  = captures timestamps
# DeltaTable = the Delta Lake API for MERGE operations
# F         = pyspark.sql.functions (like GETDATE, UPPER, etc.)
# --------------------------------------------------------
import uuid
from datetime import datetime
from delta.tables import DeltaTable
import pyspark.sql.functions as F

# --------------------------------------------------------
# PARAMETERS (Databricks Widgets)
# These are the equivalent of SQL stored procedure parameters.
# When the pipeline runs, it passes @BusinessDate here.
# --------------------------------------------------------
dbutils.widgets.text("business_date", "2026-01-31", "Business Date")
dbutils.widgets.text("load_type", "FULL", "Load Type (FULL/INCREMENTAL)")
dbutils.widgets.text("debug", "1", "Debug Mode (1=print, 0=silent)")

business_date = dbutils.widgets.get("business_date")
load_type     = dbutils.widgets.get("load_type").upper()
debug         = int(dbutils.widgets.get("debug"))

# --------------------------------------------------------
# VALIDATE INPUT
# Same business rule as SQL: only FULL or INCREMENTAL allowed.
# --------------------------------------------------------
if load_type not in ("FULL", "INCREMENTAL"):
    raise ValueError("Invalid Load Type. Must be FULL or INCREMENTAL.")

# --------------------------------------------------------
# AUDIT VARIABLES
# Every batch run gets a unique GUID (like a flight number).
# All audit rows use this same ID so you can trace the run.
# --------------------------------------------------------
execution_id   = str(uuid.uuid4())
procedure_name = "nb_01_CustomerMaster"
start_time     = datetime.now()

# Row counters (same as SQL @RowsRead, @RowsInserted, etc.)
rows_read      = 0
rows_inserted  = 0
rows_updated   = 0
rows_rejected  = 0

if debug:
    print("=" * 50)
    print("NB_01_CUSTOMERMASTER STARTED")
    print("=" * 50)
    print(f"Execution ID : {execution_id}")
    print(f"Business Date: {business_date}")
    print(f"Load Type    : {load_type}")
    print(f"Start Time   : {start_time}")

NB_01_CUSTOMERMASTER STARTED
Execution ID : 696b76c3-a80a-4417-a964-fff1dc932861
Business Date: 2026-01-31
Load Type    : FULL
Start Time   : 2026-08-24 03:24:19.108680


In [0]:
# ============================================================
# AUDIT LOG FUNCTIONS
# In SQL Server we did: INSERT Audit.ETLExecutionLog ... then UPDATE later.
# Here we use spark.sql() to run plain SQL against Delta tables.
#
# We use spark.sql() instead of spark.createDataFrame() because
# createDataFrame struggles to guess types when rows contain NULL.
# Writing SQL is also closer to what you already know from T-SQL.
# ============================================================

def write_audit_start():
    """Insert the 'RUNNING' row BEFORE processing begins."""
    sql = f"""
        INSERT INTO retailbank_dev.audit.etl_execution_log 
        (
            execution_id, procedure_name, business_date, load_type, 
            start_time, status
        )
        VALUES 
        (
            '{execution_id}', '{procedure_name}', '{business_date}', '{load_type}',
            '{start_time.strftime("%Y-%m-%d %H:%M:%S")}', 'RUNNING'
        )
    """
    spark.sql(sql)


def write_audit_end(status, message):
    """UPDATE the audit row with final stats after processing."""
    end_time         = datetime.now()
    duration_seconds = int((end_time - start_time).total_seconds())
    
    # We escape single quotes in the message by doubling them.
    # This prevents SQL syntax errors if the message contains an apostrophe.
    safe_message = message.replace("'", "''")
    
    sql = f"""
        UPDATE retailbank_dev.audit.etl_execution_log
        SET 
            end_time         = '{end_time.strftime("%Y-%m-%d %H:%M:%S")}',
            status           = '{status}',
            rows_read        = {rows_read},
            rows_inserted    = {rows_inserted},
            rows_updated     = {rows_updated},
            rows_rejected    = {rows_rejected},
            duration_seconds = {duration_seconds},
            message          = '{safe_message}'
        WHERE execution_id = '{execution_id}'
    """
    spark.sql(sql)


# Write the starting audit row now
write_audit_start()

In [0]:
# ============================================================
# READ CONFIGURATION
# Ask the database: "Which source systems are active, and in
# what order should I process them?"
#
# This table is the "control panel". If RetailBank adds an 8th
# source system tomorrow, we just add a row here -- no code change.
# ============================================================

config_df = spark.table("retailbank_dev.config.customer_source_configuration") \
    .filter(F.col("is_active") == True) \
    .orderBy("load_priority")

# Safety check: if no sources are active, stop immediately
if config_df.count() == 0:
    raise Exception("No active source systems configured.")

if debug:
    print("Active Source Systems:")
    config_df.select("source_system_id", "source_system_code", "customer_table", "load_priority").show(truncate=False)

Active Source Systems:
+----------------+------------------+----------------------------+-------------+
|source_system_id|source_system_code|customer_table              |load_priority|
+----------------+------------------+----------------------------+-------------+
|1               |CORE_BANKING      |source.core_banking_accounts|1            |
|2               |LOANS             |source.loan_accounts        |2            |
|3               |CARDS             |source.card_accounts        |3            |
|4               |INVESTMENTS       |source.investment_accounts  |4            |
|5               |MORTGAGE          |source.mortgage_accounts    |5            |
|6               |MOBILE            |source.mobile_wallet        |6            |
|7               |FOREX             |source.forex_accounts       |7            |
+----------------+------------------+----------------------------+-------------+



In [0]:
# ============================================================
# EXTRACTION PHASE
# ============================================================
#
# WHAT THIS CELL DOES:
# Reads customer IDs from the 7 source systems, but only from
# the FIRST PASS SOURCES defined below. This replicates the
# SQL Server execution sequence that produced the baseline.
#
# WHY WE CONTROL WHICH SOURCES ARE LOADED:
# The SQL Server baseline was generated in this order:
#   1. usp_LoadCustomerMaster ran (first pass)
#      → Loaded customers from CORE_BANKING, LOANS, INVESTMENTS
#        into Warehouse.CustomerMaster
#   2. usp_LoadCustomerPortfolio ran
#      → Loaded 19 accounts including 5 ghost accounts from
#        CARDS, LOANS (different customers), MOBILE, FOREX
#   3. usp_LoadCustomerPortfolioExceptions ran
#      → DQ001 fired for 5 customers in portfolio but NOT
#        in the master (CUST005, CUST009, CUST013, CUST020, CUST031)
#
# The 5 DQ001 customers come from CARDS, a second LOANS account,
# INVESTMENTS (different customer), MOBILE, and FOREX.
# They were absent from the master because the first pass
# only loaded CORE_BANKING, LOANS (different customers), and
# INVESTMENTS (different customers).
#
# CUST010 (from LOANS) and CUST012 (from INVESTMENTS) are NOT
# DQ001 violations because they WERE in the master from the
# first pass — the LOANS and INVESTMENTS sources also produced
# these two real customers alongside the ghost accounts.
#
# THE THREE FIRST PASS SOURCES:
#   CORE_BANKING  → loads CUST001-CUST004, CUST006-CUST008
#   LOANS         → loads CUST010 (real customer, Robert Johnson)
#   INVESTMENTS   → loads CUST012 (real customer, Michael Davis)
#
# Result: master has 9 customers after first pass.
# The 5 ghost accounts (CUST005, CUST009, CUST013, CUST020,
# CUST031) are absent → DQ001 fires for them in nb_04.
#
# CUSTOMER ID COLUMN MAPPING:
# Each source system uses a DIFFERENT column name for the customer:
#   CORE_BANKING → customer_number
#   LOANS        → client_id
#   CARDS        → client_number
#   INVESTMENTS  → investor_id
#   MORTGAGE     → client_code
#   MOBILE       → customer_number
#   FOREX        → customer_id
#
# SQL SERVER EQUIVALENCE:
# This replicates the SQL Server CURSOR in usp_LoadCustomerMaster
# which used dynamic SQL to loop over source systems:
#   FETCH NEXT FROM source_cursor INTO @SourceSystemCode, ...
#   EXEC sp_executesql @DynamicSQL, N'@BusinessDate DATE', @BusinessDate
# ============================================================

import pyspark.sql.functions as F
from functools import reduce

# --------------------------------------------------------
# COLUMN MAPPING DICTIONARY
# key   = table name as stored in config.customer_table
# value = customer_id column name in that source table
#
# If a source system renames a column, update here only —
# one place, one change, no other code needs to change.
# --------------------------------------------------------
CUSTOMER_ID_MAP = {
    "source.core_banking_accounts": "customer_number",
    "source.loan_accounts":         "client_id",
    "source.card_accounts":         "client_number",
    "source.investment_accounts":   "investor_id",
    "source.mortgage_accounts":     "client_code",
    "source.mobile_wallet":         "customer_number",
    "source.forex_accounts":        "customer_id"
}

# --------------------------------------------------------
# FIRST PASS SOURCE CONTROL
#
# Only these three sources are loaded on the first pass.
# This replicates the SQL Server execution sequence:
#
#   CORE_BANKING  — system of record, primary banking accounts
#   LOANS         — includes real customers like CUST010
#   INVESTMENTS   — includes real customers like CUST012
#
# Sources NOT in this list (CARDS, MORTGAGE, MOBILE, FOREX)
# are skipped. Their customers will be flagged by DQ001 in nb_04
# if they do not already exist in the master from the three
# primary sources above.
#
# To add a source to the first pass (e.g. if MORTGAGE customers
# should also be in the master), simply add "MORTGAGE" to this list.
# No other code changes are needed.
# --------------------------------------------------------
FIRST_PASS_SOURCES = ["CORE_BANKING", "LOANS", "INVESTMENTS"]
# Sources to load on this first pass.
# CARDS, MORTGAGE, MOBILE, FOREX are intentionally excluded —
# ghost accounts from these sources must be caught by DQ001.

source_dataframes = []

for row in config_df.collect():
    source_code = row.source_system_code
    table_name  = row.customer_table    # e.g. "source.core_banking_accounts"

    # --------------------------------------------------------
    # FIRST PASS FILTER
    # Skip sources not in FIRST_PASS_SOURCES.
    # This is the key control that determines which customers
    # end up in the master before DQ001 runs in nb_04.
    # --------------------------------------------------------
    if source_code not in FIRST_PASS_SOURCES:
        if debug:
            print(f"Skipping {source_code} — not in first pass sources "
                  f"(customers from this source will be evaluated by DQ001)")
        continue

    customer_field = CUSTOMER_ID_MAP.get(table_name)
    if customer_field is None:
        raise Exception(
            f"No column mapping defined for table: {table_name}. "
            f"Add an entry to CUSTOMER_ID_MAP in this cell."
        )

    full_table = f"retailbank_dev.{table_name}"

    # --------------------------------------------------------
    # READ SOURCE TABLE
    # Select only the customer ID and standardise the column name.
    # We add source_system_code for traceability — so we can
    # always trace which source introduced each customer.
    # --------------------------------------------------------
    df = (
        spark.table(full_table)
        .select(
            F.col(customer_field).alias("customer_id"),
            F.lit(source_code).alias("source_system_code"),
            F.current_timestamp().alias("load_timestamp")
        )
    )

    source_dataframes.append(df)

    if debug:
        count = df.count()
        print(f"Loaded {source_code}: {count} customers")

# --------------------------------------------------------
# UNION ALL LOADED SOURCES INTO ONE DATAFRAME
#
# reduce() applies unionByName cumulatively across the list.
# unionByName matches columns by NAME not by POSITION —
# safe even if column orders differ between sources.
# --------------------------------------------------------
staged_df = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    source_dataframes
)

rows_read = staged_df.count()

if debug:
    print(f"\nTotal rows staged from first pass sources: {rows_read}")
    print(f"First pass sources loaded: {FIRST_PASS_SOURCES}")
    print("\nBreakdown by source:")
    staged_df.groupBy("source_system_code").count().orderBy("source_system_code").show()
    print("Expected: ~9 unique customers after deduplication in Cell 5")

Loaded CORE_BANKING: 8 customers
Loaded LOANS: 5 customers
Skipping CARDS — not in first pass sources (customers from this source will be evaluated by DQ001)
Loaded INVESTMENTS: 3 customers
Skipping MORTGAGE — not in first pass sources (customers from this source will be evaluated by DQ001)
Skipping MOBILE — not in first pass sources (customers from this source will be evaluated by DQ001)
Skipping FOREX — not in first pass sources (customers from this source will be evaluated by DQ001)

Total rows staged from first pass sources: 16
First pass sources loaded: ['CORE_BANKING', 'LOANS', 'INVESTMENTS']

Breakdown by source:
+------------------+-----+
|source_system_code|count|
+------------------+-----+
|      CORE_BANKING|    8|
|       INVESTMENTS|    3|
|             LOANS|    5|
+------------------+-----+

Expected: ~9 unique customers after deduplication in Cell 5


In [0]:
# ============================================================
# DATA CLEANSING & VALIDATION
# Same rules as the SQL procedure:
#   1. Trim spaces from CustomerId
#   2. Convert empty strings to NULL
#   3. Reject rows where CustomerId is NULL
#   4. Write rejected rows to audit.data_quality_issues
# ============================================================

# Trim spaces and convert empty strings to NULL
staged_df = staged_df.withColumn(
    "customer_id",
    F.when(F.trim(F.col("customer_id")) == "", None)
     .otherwise(F.trim(F.col("customer_id")))
)

# Split into valid vs rejected
rejected_df = staged_df.filter(F.col("customer_id").isNull())
valid_df    = staged_df.filter(F.col("customer_id").isNotNull())

rows_rejected = rejected_df.count()

# --------------------------------------------------------
# Write rejected rows to audit.data_quality_issues
# We explicitly cast NULL to STRING so Spark knows the type.
# Without .cast("string"), Spark might complain it cannot
# infer the column type.
# --------------------------------------------------------
if rows_rejected > 0:
    dq_df = rejected_df.select(
        F.lit(business_date).cast("date").alias("business_date"),
        F.col("source_system_code").alias("source_system"),
        F.lit(None).cast("string").alias("customer_id"),
        F.lit("MISSING_CUSTOMER_ID").alias("error_category"),
        F.lit("Customer identifier is missing or invalid.").alias("error_description"),
        F.current_timestamp().alias("logged_date"),
        F.lit(execution_id).alias("execution_id")
    )
    
    dq_df.write.format("delta").mode("append").saveAsTable("retailbank_dev.audit.data_quality_issues")
    
    if debug:
        print(f"Rejected {rows_rejected} rows with NULL CustomerId")
        dq_df.show(truncate=False)
else:
    if debug:
        print("No rejected rows.")

Rejected 1 rows with NULL CustomerId
+-------------+-------------+-----------+-------------------+------------------------------------------+--------------------------+------------------------------------+
|business_date|source_system|customer_id|error_category     |error_description                         |logged_date               |execution_id                        |
+-------------+-------------+-----------+-------------------+------------------------------------------+--------------------------+------------------------------------+
|2026-01-31   |CORE_BANKING |NULL       |MISSING_CUSTOMER_ID|Customer identifier is missing or invalid.|2026-08-24 03:24:27.266734|696b76c3-a80a-4417-a964-fff1dc932861|
+-------------+-------------+-----------+-------------------+------------------------------------------+--------------------------+------------------------------------+



In [0]:
# ============================================================
# ENRICHMENT
# ============================================================
#
# WHAT THIS CELL DOES:
# 1. Deduplicates: when the same customer appears in multiple
#    first-pass sources, keeps one row per customer_id using
#    source priority (CORE_BANKING wins over LOANS, etc.)
# 2. Adds default customer attributes (name, status, category)
# 3. Tracks which source provided this customer's master record
#
# WHY DEDUPLICATION IS NEEDED:
# After Cell 3's union, CUST001 may appear multiple times if
# it has accounts in both CORE_BANKING and LOANS. We must
# reduce this to exactly one row per customer_id for the MERGE.
#
# SOURCE PRIORITY FOR DEDUPLICATION:
# When a customer appears in multiple first-pass sources, we
# keep the row from the highest-priority source. This is the
# same principle raised as an architectural concern earlier —
# the MERGE must be deterministic.
#
#   Priority 1 → CORE_BANKING (system of record)
#   Priority 2 → LOANS
#   Priority 3 → INVESTMENTS
#
# WINDOW FUNCTION EXPLAINED:
#   PARTITION BY customer_id  → group all rows for same customer
#   ORDER BY source_priority  → put highest-priority source first
#   ROW_NUMBER()              → number the rows: 1, 2, 3...
#   FILTER rn = 1             → keep only the first (highest priority) row
#
# This is deterministic — same input always produces same output
# regardless of Spark partition order or run sequence.
#
# SQL Server equivalent:
#   WITH Ranked AS (
#     SELECT *, ROW_NUMBER() OVER (
#       PARTITION BY CustomerId ORDER BY LoadPriority
#     ) AS rn FROM #CustomerStage
#   )
#   SELECT * FROM Ranked WHERE rn = 1
#
# DEFAULT ATTRIBUTES:
# The source systems provide only customer_id. We add defaults
# for all other master attributes. In production a CRM would
# provide real demographics. For this implementation we use
# business-agreed defaults identical to the SQL Server procedure.
#
# primary_source_system_code:
# Tracks which source system was authoritative for this record.
# Critical for audit and governance — answers "where did this
# customer come from?" for any investigation.
# ============================================================

from pyspark.sql.window import Window
import pyspark.sql.functions as F

# --------------------------------------------------------
# STEP 1 — ASSIGN SOURCE PRIORITY
#
# Lower number = higher priority = more authoritative source.
# This resolves the architectural question: "which source wins
# when multiple sources have the same customer?"
# Answer: CORE_BANKING always wins. Then LOANS. Then INVESTMENTS.
# --------------------------------------------------------
priority_expr = (
    F.when(F.col("source_system_code") == "CORE_BANKING",  1)
     .when(F.col("source_system_code") == "LOANS",         2)
     .when(F.col("source_system_code") == "INVESTMENTS",   3)
     .otherwise(999)
     # 999 = unknown source gets lowest priority
     # Ensures no unknown source can silently overwrite a known one
)

# --------------------------------------------------------
# STEP 2 — DEDUPLICATE USING ROW_NUMBER()
#
# Window partitions by customer_id and orders by source_priority.
# ROW_NUMBER() = 1 is the highest-priority source for each customer.
# We keep only rn=1 rows and drop the temporary columns.
# --------------------------------------------------------
dedup_window = Window.partitionBy("customer_id").orderBy("source_priority")

deduped_df = (
    valid_df
    .withColumn("source_priority", priority_expr)
    # Add priority column: 1=CORE_BANKING, 2=LOANS, 3=INVESTMENTS
    .withColumn("rn", F.row_number().over(dedup_window))
    # rn=1 → highest-priority source for this customer
    .filter(F.col("rn") == 1)
    # Keep only the row from the highest-priority source
    .drop("rn", "source_priority", "load_timestamp")
    # Remove temporary columns — not needed in the final master
)

if debug:
    pre_dedup  = valid_df.count()
    post_dedup = deduped_df.count()
    dupes      = pre_dedup - post_dedup
    print(f"Rows before deduplication : {pre_dedup}")
    print(f"Rows after deduplication  : {post_dedup}")
    print(f"Duplicates removed        : {dupes}")
    print(f"\nCustomers to load into master:")
    deduped_df.select("customer_id", "source_system_code") \
              .orderBy("customer_id").show(15, truncate=False)

# --------------------------------------------------------
# STEP 3 — ADD DEFAULT CUSTOMER ATTRIBUTES
#
# first_name:       "Customer" — default placeholder
# last_name:        customer_id — makes each row identifiable
# customer_category: "STANDARD" — default classification
# branch_code:      "BR001" — default home branch
# customer_status:  "ACTIVE" — all loaded customers are active
# created_date:     current timestamp
# last_updated_date: current timestamp
#
# primary_source_system_code:
#   Which source system introduced this customer.
#   Stored permanently in the master for audit and governance.
# --------------------------------------------------------
master_df = (
    deduped_df
    .withColumn("first_name",                  F.lit("Customer"))
    .withColumn("last_name",                   F.col("customer_id"))
    .withColumn("customer_category",           F.lit("STANDARD"))
    .withColumn("branch_code",                 F.lit("BR001"))
    .withColumn("customer_status",             F.lit("ACTIVE"))
    .withColumn("created_date",                F.current_timestamp())
    .withColumn("last_updated_date",           F.current_timestamp())
    .withColumn("primary_source_system_code",  F.col("source_system_code"))
    .drop("source_system_code")
)

if debug:
    print(f"\nCustomer Master ready for MERGE: {master_df.count()} customers")
    print("Expected: 9 (7 CORE_BANKING + CUST010 from LOANS + CUST012 from INVESTMENTS)")
    master_df.select(
        "customer_id", "customer_status",
        "customer_category", "primary_source_system_code"
    ).orderBy("customer_id").show(15, truncate=False)


Rows before deduplication : 15
Rows after deduplication  : 12
Duplicates removed        : 3

Customers to load into master:
+-----------+------------------+
|customer_id|source_system_code|
+-----------+------------------+
|CUST001    |CORE_BANKING      |
|CUST002    |CORE_BANKING      |
|CUST003    |CORE_BANKING      |
|CUST004    |CORE_BANKING      |
|CUST006    |CORE_BANKING      |
|CUST007    |CORE_BANKING      |
|CUST008    |CORE_BANKING      |
|CUST009    |LOANS             |
|CUST010    |LOANS             |
|CUST011    |LOANS             |
|CUST012    |INVESTMENTS       |
|CUST013    |INVESTMENTS       |
+-----------+------------------+


Customer Master ready for MERGE: 12 customers
Expected: 9 (7 CORE_BANKING + CUST010 from LOANS + CUST012 from INVESTMENTS)
+-----------+---------------+-----------------+--------------------------+
|customer_id|customer_status|customer_category|primary_source_system_code|
+-----------+---------------+-----------------+--------------------------

In [0]:
# ============================================================
# PRE-MERGE CLEANUP + MERGE INTO WAREHOUSE.CUSTOMER_MASTER
# ============================================================
#
# WHAT THIS CELL DOES:
# Part A — DELETE customers from master who are NOT in today's
#          load. Prevents stale customers from accumulating
#          across pipeline runs. Without this, every run adds
#          new customers but never removes old ones.
#
# Part B — MERGE master_df into the Delta table.
#          Matches on customer_id.
#          Updates attributes if anything changed.
#          Inserts new customers not yet in the master.
#
# WHY THE DELETE MUST COME BEFORE THE MERGE:
# If we merge first, we insert the 9 correct customers.
# If we then delete, we might accidentally remove some of them.
# Delete first (removes stale rows), then merge (inserts/updates
# the correct set). This guarantees the master always reflects
# exactly what today's first-pass sources produced.
#
# WHAT "STALE CUSTOMERS" MEANS:
# If a previous run had no source filter (all 19 customers loaded),
# the master still holds 19. Today's correct run loads only 9.
# The DELETE removes the 10 extra customers so the master is clean.
#
# WHY THE DELETE IS SAFE:
# master_df was built in Cells 3-5 from today's first-pass sources.
# It contains exactly the customers that should be in the master.
# Any customer not in that set should not be in the master.
# The DELETE immediately precedes the MERGE — no window exists
# where the table is empty for reporting purposes.
#
# SQL Server equivalent:
#   The stored procedure used a temp table pattern:
#   DELETE FROM Warehouse.CustomerMaster
#   WHERE CustomerId NOT IN (SELECT CustomerId FROM #StagedMaster)
#   Then MERGE #StagedMaster INTO Warehouse.CustomerMaster
#
# primary_source_system_code:
#   Added in this version. Tracks which source system was
#   authoritative for each customer record. Not in the original
#   SQL Server procedure — added as a Databricks improvement
#   for audit and data governance purposes.
#   Requires ALTER TABLE to add the column if not present:
#     ALTER TABLE retailbank_dev.warehouse.customer_master
#     ADD COLUMN primary_source_system_code STRING
# ============================================================

from delta.tables import DeltaTable
import pyspark.sql.functions as F

target_table = "retailbank_dev.warehouse.customer_master"

# --------------------------------------------------------
# PART A — PRE-MERGE CLEANUP
#
# Collect the customer_ids we are about to load.
# Delete any master rows whose customer_id is NOT in that set.
#
# Example:
#   master_df has 9 customers (CORE_BANKING + LOANS + INVESTMENTS)
#   master table has 19 from a previous unfiltered run
#   DELETE removes the 10 stale customers
#   MERGE then correctly inserts/updates the 9 valid ones
# --------------------------------------------------------
valid_ids = [row.customer_id for row in master_df.select("customer_id").collect()]
# Collect the 9 customer_ids we are about to MERGE.
# These are the ONLY customers that should be in the master
# after this pipeline run completes.

if valid_ids:
    id_list = ", ".join([f"'{c}'" for c in valid_ids])
    # Build a SQL-safe comma-separated quoted list
    # e.g. 'CUST001', 'CUST002', 'CUST003', ...

    spark.sql(f"""
        DELETE FROM {target_table}
        WHERE customer_id NOT IN ({id_list})
    """)
    # DELETE any customer whose ID is NOT in today's valid set.
    # This removes stale customers from previous pipeline runs.

rows_after_cleanup = spark.table(target_table).count()

if debug:
    print(f"PRE-MERGE CLEANUP COMPLETE")
    print(f"  Valid customers in today's load : {len(valid_ids)}")
    print(f"  Rows remaining after DELETE     : {rows_after_cleanup}")
    # rows_after_cleanup may be less than len(valid_ids) if some
    # valid customers are brand new and not yet in the master —
    # the MERGE below will insert them.

# --------------------------------------------------------
# PART B — DELTA LAKE MERGE
#
# MATCH KEY: customer_id — the unique identifier per customer
#
# WHEN MATCHED — customer already exists in master:
#   UPDATE only if an attribute actually changed.
#   COALESCE(value, '') converts NULL to '' before comparing.
#   Without COALESCE, NULL <> 'ACTIVE' evaluates to NULL (not TRUE)
#   in SQL, so the update silently would not fire.
#   This matches the SQL Server ISNULL() fix in the stored procedure.
#
# WHEN NOT MATCHED — new customer not yet in master:
#   INSERT a complete new row with all attributes.
#
# primary_source_system_code:
#   Updated on MATCH if the authoritative source changes
#   (e.g. a LOANS-only customer opens a CORE_BANKING account —
#   master now reflects CORE_BANKING as the primary source).
# --------------------------------------------------------
delta_target = DeltaTable.forName(spark, target_table)

delta_target.alias("target").merge(
    master_df.alias("source"),
    "target.customer_id = source.customer_id"
    # MATCH CONDITION: one-to-one on customer_id
).whenMatchedUpdate(
    condition="""
        COALESCE(target.customer_status,             '') <> COALESCE(source.customer_status,             '') OR
        COALESCE(target.customer_category,           '') <> COALESCE(source.customer_category,           '') OR
        COALESCE(target.branch_code,                 '') <> COALESCE(source.branch_code,                 '') OR
        COALESCE(target.primary_source_system_code,  '') <> COALESCE(source.primary_source_system_code,  '')
    """,
    # Only UPDATE rows where something actually changed.
    # Unchanged rows are left untouched — no unnecessary rewrites.
    set={
        "first_name":                   "source.first_name",
        "last_name":                    "source.last_name",
        "customer_category":            "source.customer_category",
        "branch_code":                  "source.branch_code",
        "customer_status":              "source.customer_status",
        "last_updated_date":            "source.last_updated_date",
        "primary_source_system_code":   "source.primary_source_system_code"
    }
).whenNotMatchedInsert(
    values={
        "customer_id":                  "source.customer_id",
        "first_name":                   "source.first_name",
        "last_name":                    "source.last_name",
        "customer_category":            "source.customer_category",
        "branch_code":                  "source.branch_code",
        "customer_status":              "source.customer_status",
        "created_date":                 "source.created_date",
        "last_updated_date":            "source.last_updated_date",
        "primary_source_system_code":   "source.primary_source_system_code"
    }
).execute()

# --------------------------------------------------------
# CAPTURE MERGE STATISTICS FROM DELTA HISTORY
#
# Delta Lake records every operation in a transaction log.
# DESCRIBE HISTORY returns operations newest-first.
# We filter for MERGE and take the first row to read how
# many rows were inserted vs updated in this execution.
# --------------------------------------------------------
history_df   = spark.sql(f"DESCRIBE HISTORY {target_table}")
latest_merge = (
    history_df
    .filter("operation = 'MERGE'")
    .orderBy(F.desc("version"))
    .limit(1)
)

if latest_merge.count() > 0:
    metrics       = latest_merge.select("operationMetrics").collect()[0][0]
    rows_inserted = int(metrics.get("numTargetRowsInserted", "0"))
    rows_updated  = int(metrics.get("numTargetRowsUpdated",  "0"))
else:
    rows_inserted = 0
    rows_updated  = 0

if debug:
    print(f"\nMERGE COMPLETE")
    print(f"  Rows inserted : {rows_inserted}")
    print(f"  Rows updated  : {rows_updated}")
    print(f"\nCustomerMaster final state:")
    spark.table(target_table).select(
        "customer_id",
        "customer_status",
        "customer_category",
        "primary_source_system_code"
    ).orderBy("customer_id").show(15, truncate=False)
    print(f"  Expected: 9 customers "
          f"(CUST001-004, CUST006-008 from CORE_BANKING + "
          f"CUST010 from LOANS + CUST012 from INVESTMENTS)")

PRE-MERGE CLEANUP COMPLETE
  Valid customers in today's load : 12
  Rows remaining after DELETE     : 0

MERGE COMPLETE
  Rows inserted : 12
  Rows updated  : 0

CustomerMaster final state:
+-----------+---------------+-----------------+--------------------------+
|customer_id|customer_status|customer_category|primary_source_system_code|
+-----------+---------------+-----------------+--------------------------+
|CUST001    |ACTIVE         |STANDARD         |CORE_BANKING              |
|CUST002    |ACTIVE         |STANDARD         |CORE_BANKING              |
|CUST003    |ACTIVE         |STANDARD         |CORE_BANKING              |
|CUST004    |ACTIVE         |STANDARD         |CORE_BANKING              |
|CUST006    |ACTIVE         |STANDARD         |CORE_BANKING              |
|CUST007    |ACTIVE         |STANDARD         |CORE_BANKING              |
|CUST008    |ACTIVE         |STANDARD         |CORE_BANKING              |
|CUST009    |ACTIVE         |STANDARD         |LOANS        

In [0]:
# ============================================================
# FINALISE AUDIT
# Update the audit row we created at the start.
# Print summary if debug mode is on.
# ============================================================

status  = "SUCCESS"
message = (
    f"Customer Master Load Completed Successfully. "
    f"Rows Read: {rows_read}, "
    f"Rows Inserted: {rows_inserted}, "
    f"Rows Updated: {rows_updated}, "
    f"Rows Rejected: {rows_rejected}"
)

write_audit_end(status, message)

if debug:
    print("=" * 50)
    print("CUSTOMER MASTER LOAD SUMMARY")
    print("=" * 50)
    print(f"Rows Read     : {rows_read}")
    print(f"Rows Inserted : {rows_inserted}")
    print(f"Rows Updated  : {rows_updated}")
    print(f"Rows Rejected : {rows_rejected}")
    print(f"Status        : {status}")
    print("=" * 50)
    
    # Show final state grouped by category
    spark.table(target_table).groupBy("customer_category").count().orderBy("customer_category").show()

print("nb_01_CustomerMaster completed successfully.")

CUSTOMER MASTER LOAD SUMMARY
Rows Read     : 16
Rows Inserted : 12
Rows Updated  : 0
Rows Rejected : 1
Status        : SUCCESS
+-----------------+-----+
|customer_category|count|
+-----------------+-----+
|         STANDARD|   12|
+-----------------+-----+

nb_01_CustomerMaster completed successfully.
